# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [73]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [74]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2:3b'
ollama_base_url = "http://localhost:11434/v1"
openai = OpenAI(base_url = ollama_base_url, api_key=api_key)

API key looks good so far


In [75]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [76]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [77]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [78]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [79]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [80]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'About page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Company page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'Home/Principal link (again since home is a directory of sorts)',
   'url': 'https://edwarddonner.com'},
  {'type': 'Blog posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page (profile)',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [82]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [83]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 1 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [84]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:3b
Found 9 relevant links


{'links': [{'type': 'Company/Enterprise Page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'Models Page', 'url': 'https://huggingface.co/models'},
  {'type': 'Dataset Page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'Pricing Page', 'url': 'https://huggingface.co/pricing#spaces'},
  {'type': 'Docs/Transformers Page',
   'url': 'https://huggingface.co/docs/transformers'},
  {'type': 'Docs/Diffusers Page',
   'url': 'https://huggingface.co/docs/diffusers'},
  {'type': 'Docs/Safetensors Page',
   'url': 'https://huggingface.co/docs/safetensors'},
  {'type': 'Company/About Page', 'url': 'https://huggingface.co/about'},
  {'type': 'Join Careers Page', 'url': 'https://huggingface.co/join'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [85]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [86]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 2 relevant links
## Landing Page:

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy

In [87]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [88]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [89]:
get_brochure_user_prompt("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 2 relevant links


'\nYou are looking at a company called: Edward Donner\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 2021\n.\nI will ha

In [90]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [91]:
create_brochure("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 3 relevant links


# Edward Donner: Empowering People through AI-Driven Discovery

## About Us

At Edward Donner, we're passionate about harnessing the power of Artificial Intelligence (AI) to unlock human potential and drive transformational change. Our mission is to empower individuals to discover their purpose, pursue their passion, and find fulfillment in their careers.

 Founded by Ed, our CTO and AI expert, Edward Donner is built on a foundation of innovation, collaboration, and a deep understanding of the human spirit. With years of experience in AI and a proven track record of creating impactful technologies, we're committed to pushing the boundaries of what's possible with machine learning.

## Our Story

Ed's journey began in 2013 when he founded an AI startup that would eventually evolve into Nebula.io. After successfully acquiring the company in 2021, Ed refocused his efforts on building a platform that could positively impact people's lives. The result is Edward Donner, a cutting-edge technology company that's redefining the recruitment industry with its patented Generative AI model.

## Our Focus

At Edward Donner, we're focused on three core areas:

*   **Empowering Human Potential**: We believe that everyone has unique talents and abilities waiting to be unleashed. Our platform helps individuals discover their strengths and passions, empowering them to make informed decisions about their careers.
*   **Transformative Technology**: Our patented AI model is revolutionizing the recruitment industry by providing a more accurate and efficient way to match candidates with roles. We're committed to continuously improving our technology to stay ahead of the curve.
*   **Positive Impact**: We're driven by a mission to make a positive impact on people's lives. By empowering individuals to pursue their dream jobs, we're contributing to a world that values purpose, fulfillment, and human prosperity.

## Join Our Community

If you're passionate about AI, innovation, and making a difference in people's lives, we invite you to join our community. Whether you're an aspiring developer, a seasoned professional looking for new opportunities, or simply someone eager to learn more about the latest advancements in tech, we've got something for everyone.

## Get In Touch

Stay up-to-date with our latest news, updates, and insights into the world of AI and recruitment by following us on [LinkedIn](#) , [Twitter](#).

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [92]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [93]:
stream_brochure("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 5 relevant links


# Edward Donner: Revolutionizing Human Potential through AI


## Mission and Vision


At Edward Donner, we believe that AI has the power to revolutionize human potential by helping people discover their reason for being and pursue their most fulfilling careers. Our mission is to empower individuals, businesses, and society as a whole with cutting-edge Generative AI solutions.


## About Us

We're led by Ed Donner, a passionate advocate for applying AI to real-world problems. With his expertise in machine learning and leadership experience from founding and leading successful AI startups, Ed is committed to driving innovation that makes a positive impact on people's lives.


## Our Focus


Our focus areas include:

*   **AI Recruiting**: Using patented models to match people with roles where they can thrive
*   **Ikigai Identification**: Helping individuals discover their purpose in life
*   **AI Adoption**: Providing tools and resources for businesses to leverage AI solutions effectively


## Community Engagement


We're dedicated to building a supportive community around our mission. Join us for live events, workshops, and online discussions on the latest AI trends and applications.


## Stay Connected

Stay up-to-date with the latest news, insights, and updates from Edward Donner by following us on social media or subscribing to our newsletter.

### Contact Us
`ed [at] edwarddonner [dot] com`
`www.edwarddonner.com`

Join the movement to unlock human potential through AI.

In [94]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:3b
Found 3 relevant links


# Hugging Face: Empowering the Future of Machine Learning

## About Us

Hugging Face is the premier platform for machine learning, connecting innovators and industry leaders to unlock breakthroughs in AI. Our mission is to accelerate the development and deployment of artificial intelligence, empowering humans and machines to collaborate and augment each other.

## What We Offer

Our innovative platform provides a suite of tools and services designed to simplify the process of building, deploying, and managing AI models. With Hugging Face, developers can:

* Easily deploy their AI models to production on our fully managed platform
* Browse a vast catalog of pre-built, ready-to-deploy models for popular AI technologies like LLaMA, T5, and BERT
* Access cutting-edge infrastructure and expertise to accelerate model performance and scalability

## Our Community and Culture

We are proud to be part of the thriving machine learning ecosystem, fostering collaboration and innovation among millions of users worldwide. Our community-driven platform is built on openness, transparency, and a passion for accelerating AI research.

Our culture values creativity, empathy, and resilience over traditional corporate hierarchies. We believe in empowering individual contributions and fostering an inclusive environment where diverse perspectives and experiences thrive.

## Partnerships and Impact

Hugging Face has established strategic partnerships with leading technology companies to accelerate the development of AI solutions. Our platform is used by top organizations across industries to drive innovation, efficiency, and customer satisfaction.

Our mission-driven approach extends beyond our community; we are committed to reducing digital inequality and promoting economic growth through responsible AI adoption. By empowering a global generation of AI innovators, we will help shape a more equitable and sustainable future.

## Career Opportunities

At Hugging Face, we are passionate about building a talented team that shares our vision for accelerating machine learning innovation. We welcome applications from experienced professionals, recent graduates, and budding talent seeking to contribute to the development of cutting-edge AI technologies.

Explore job opportunities on our careers page and join the revolution: [link to careers page]

## Contact Us

Innovate with us! Learn more about Hugging Face's mission-driven approach by visiting our website or contacting us at [huggingface@example.com](mailto:huggingface@example.com)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>